<div align="center">

# Universidad de Sevilla  
## Grado en Ingeniería del Software  
### Escuela Técnica Superior de Ingeniería Informática  
### Inteligencia Artificial (IA) – Curso 2024/2025  

<img src="img/portada.png" width="300"/>

---

#  Ensamble Secuencial de Modelos Predictivos  
### Primera Convocatoria – Junio 2025  

**Profesor:** Juan Galán Páez  
**Autores:**  
Carlos Martín de Prado Barragán  
Marco Padilla Gómez  

**Fecha de entrega:** 9 de Junio de 2025  

</div>

## Introducción

El objetivo de este notebook es implementar y evaluar un meta-modelo de ensamble secuencial para tareas de regresión, utilizando el conjunto de datos sobre la enfermedad de Parkinson. Este tipo de modelo se construye mediante la combinación iterativa de varios modelos base, donde cada nuevo modelo aprende a corregir los errores residuales cometidos por los anteriores.

Se emplea el `DecisionTreeRegressor` como modelo base, y se utilizan técnicas de preprocesamiento como la estandarización de variables para mejorar el rendimiento del entrenamiento. La evaluación se realiza mediante el coeficiente de determinación R², que mide la proporción de variabilidad explicada por el modelo.

Este enfoque está inspirado en técnicas de boosting y tiene como finalidad mejorar la capacidad predictiva frente a modelos individuales, proporcionando un marco flexible y escalable para resolver problemas de regresión complejos.

## 1. Importación de librerías 

En esta sección se importan todas las librerías necesarias para el proyecto:

- `numpy` y `pandas` para manipulación y análisis de datos.
- Módulos de `scikit-learn` para:
  - Modelos de regresión (`DecisionTreeRegressor`).
  - Preprocesamiento de datos (`StandardScaler`).
  - División de datos en entrenamiento y prueba (`train_test_split`).
  - Métricas de evaluación (`r2_score`).
- Además, se importa el meta-modelo `SequentialEnsembleRegressor` que implementamos en el módulo `sequential_ensemble.py`.

In [3]:
# Importación de librerías necesarias
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

# Importar el meta-modelo desde nuestro módulo
from sequential_ensemble import SequentialEnsembleRegressor

## 2. Carga y exploración inicial del dataset

En esta sección se carga el conjunto de datos `parkinsons.csv` desde la carpeta `data`.  
A continuación, se muestra información general del DataFrame, incluyendo:

- El número de filas y columnas.
- El tipo de datos de cada columna.
- Los primeros registros del conjunto para una inspección visual rápida.

In [4]:
# Cargar el dataset desde la carpeta 'data'
df = pd.read_csv("data/parkinsons.csv")

# Mostrar información general
df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 20 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   age            2000 non-null   int64  
 1   sex            2000 non-null   int64  
 2   test_time      2000 non-null   float64
 3   Jitter(%)      2000 non-null   float64
 4   Jitter(Abs)    2000 non-null   float64
 5   Jitter:RAP     2000 non-null   float64
 6   Jitter:PPQ5    2000 non-null   float64
 7   Jitter:DDP     2000 non-null   float64
 8   Shimmer        2000 non-null   float64
 9   Shimmer(dB)    2000 non-null   float64
 10  Shimmer:APQ3   2000 non-null   float64
 11  Shimmer:APQ5   2000 non-null   float64
 12  Shimmer:APQ11  2000 non-null   float64
 13  Shimmer:DDA    2000 non-null   float64
 14  NHR            2000 non-null   float64
 15  HNR            2000 non-null   float64
 16  RPDE           2000 non-null   float64
 17  DFA            2000 non-null   float64
 18  PPE     

,age,sex,test_time,Jitter(%),Jitter(Abs),Jitter:RAP,Jitter:PPQ5,Jitter:DDP,Shimmer,Shimmer(dB),Shimmer:APQ3,Shimmer:APQ5,Shimmer:APQ11,Shimmer:DDA,NHR,HNR,RPDE,DFA,PPE,total_UPDRS
0,49,0,194.890,0.01674,0.000156,0.01055,0.00750,0.03166,0.09713,0.833,0.05832,0.04904,0.06635,0.17497,0.194060,12.612,0.76658,0.66783,0.44528,25.011
1,66,1,135.410,0.00520,0.000029,0.00179,0.00220,0.00536,0.05364,0.471,0.03006,0.02792,0.03804,0.09019,0.065038,15.346,0.63731,0.54549,0.25823,32.000
2,63,0,61.338,0.00548,0.000043,0.00259,0.00259,0.00776,0.03919,0.353,0.02147,0.02547,0.03266,0.06440,0.017563,21.109,0.56231,0.64180,0.20707,39.674
3,75,0,181.350,0.00701,0.000055,0.00270,0.00371,0.00810,0.02869,0.269,0.01196,0.01599,0.02768,0.03589,0.017749,20.992,0.50528,0.76762,0.24917,24.925
4,55,0,168.840,0.01601,0.000113,0.00883,0.00775,0.02650,0.04998,0.494,0.02814,0.02878,0.03887,0.08443,0.091321,17.524,0.60864,0.73426,0.47043,26.230


## 3. Preprocesamiento de los datos

En este paso se realiza la preparación de los datos para el entrenamiento:

- Se separan las variables predictoras (`X`) y la variable objetivo (`y`), que en este caso es `total_UPDRS`.
- Se aplica una **estandarización** a las variables predictoras usando `StandardScaler` para normalizar sus valores, poniendo media 0 y desviación estándar 1. Esto mejora el rendimiento y estabilidad del modelo.
- El conjunto de datos estandarizado se divide en entrenamiento (80%) y prueba (20%) para evaluar el modelo de forma objetiva.
- Se convierte `y_train` a un array de NumPy para evitar problemas de indexación con pandas durante el entrenamiento del meta-modelo.

In [7]:
# Separar variables predictoras y variable objetivo
X = df.drop(columns=["total_UPDRS"])
y = df["total_UPDRS"]

# Estandarización de las variables predictoras
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Dividir en conjunto de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Evitar error de pandas vs numpy: convertir y_train a array
y_train = y_train.to_numpy()

## 4. Creación y entrenamiento del meta-modelo secuencial

En esta sección se crea una instancia del `SequentialEnsembleRegressor` utilizando como estimador base un árbol de decisión (`DecisionTreeRegressor`).  
Los hiperparámetros configurados son:

- `n_estimators=50`: se entrenan 50 modelos secuenciales para mejorar la predicción.
- `lr=0.1`: tasa de aprendizaje que modula la influencia de cada modelo individual.
- `sample_size=0.75`: cada modelo se entrena con el 75% de los datos seleccionados aleatoriamente sin reemplazo.
- `est_params={"max_depth": 4}`: se limita la profundidad máxima del árbol para controlar su complejidad.
- `random_state=42`: semilla para asegurar la reproducibilidad del experimento.

Finalmente, se entrena el meta-modelo con los datos de entrenamiento escalados y preparados.

In [8]:
# Crear y entrenar el meta-modelo con árboles de decisión
modelo = SequentialEnsembleRegressor(
    base_estimator=DecisionTreeRegressor,
    n_estimators=50,
    lr=0.1,
    sample_size=0.75,
    est_params={"max_depth": 4},
    random_state=42
)

modelo.fit(X_train, y_train)

SequentialEnsembleRegressor(base_estimator=<class 'sklearn.tree._classes.DecisionTreeRegressor'>,
                            est_params={'max_depth': 4}, n_estimators=50,
                            random_state=42, sample_size=0.75)

## 5. Predicción y evaluación del modelo

En esta última sección se generan las predicciones del meta-modelo sobre el conjunto de prueba (`X_test`).  
A continuación, se evalúa la calidad de las predicciones calculando el coeficiente de determinación R², que mide qué proporción de la variabilidad en la variable objetivo es explicada por el modelo.

Un valor de R² cercano a 1 indica un buen ajuste del modelo a los datos de prueba.

In [15]:
# Generar predicciones para análisis y posibles visualizaciones
y_pred = modelo.predict(X_test)

# Evaluar con el método score() para simplicidad
r2 = modelo.score(X_test, y_test)

print(f"R² en test: {r2:.4f}")
print("El modelo explica el {:.2f}% de la variabilidad de los datos de prueba.".format(r2*100))

R² en test: 0.7705
El modelo explica el 77.05% de la variabilidad de los datos de prueba.


## 6. Conclusiones

- Se ha implementado y entrenado un meta-modelo de ensamble secuencial basado en árboles de decisión, capaz de corregir iterativamente los errores de predicción.
- La estandarización de las variables predictoras contribuyó a un entrenamiento estable y efectivo del modelo.
- El modelo alcanzó un coeficiente de determinación R² de aproximadamente 0.77, lo que indica que explica el 77% de la variabilidad en los datos de prueba del conjunto Parkinson.
- Esta metodología permite mejorar el rendimiento respecto a modelos individuales simples al combinar múltiples modelos especializados en errores residuales.
- En siguientes etapas, se explorará el ajuste de hiperparámetros, validación cruzada y la aplicación a otros conjuntos de datos para consolidar la robustez y generalización del modelo.